In [1]:
!git clone https://github.com/Ibraheem-Al-hafith/miniGPT.git
%cd miniGPT

Cloning into 'miniGPT'...
remote: Enumerating objects: 206, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 206 (delta 9), reused 12 (delta 4), pack-reused 169 (from 1)
Receiving objects: 100% (206/206), 2.35 MiB | 8.77 MiB/s, done.
Resolving deltas: 100% (100/100), done.
/content/miniGPT


In [ ]:
import pandas as pd
 
def create_balanced_dataset(df):
    num_spam = df[df["Label"] == "spam"].shape[0]
    # Counts the instances
    ham_subset = df[df["Label"] == "ham"].sample(
    #of “spam”
    num_spam, random_state=123
    )
    # Randomly samples “ham”
    balanced_df = pd.concat([
    # instances to match the number
    ham_subset, df[df["Label"] == "spam"]
    # of “spam” instances
    ])
    return balanced_df

# Updated separator to tab
df = pd.read_csv("/content/miniGPT/SMSSpamCollection.csv", sep="\t", names=["Label", "Text"])
balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

Label
ham     747
spam    747
Name: count, dtype: int64


In [25]:
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})

In [26]:
def random_split(df, train_frac, validation_frac):
    # Shuffles the entire
    # DataFrame
    df = df.sample(
    frac=1, random_state=123
    # Calculates
    ).reset_index(drop=True)
    # split indices
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)
    # Splits the DataFrame
    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]
    return train_df, validation_df, test_df
train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)

In [27]:
train_df.to_csv("train.csv", index=None)
validation_df.to_csv("validation.csv", index=None)
test_df.to_csv("test.csv", index=None)

In [32]:
import torch
from torch.utils.data import Dataset
class SpamDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        self.data = pd.read_csv(csv_file)
        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["Text"]
        ]
        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]
        self.encoded_texts = [
            encoded_text + [pad_token_id] *
            (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]
    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length

In [33]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

train_dataset = SpamDataset(
    csv_file="train.csv",
    max_length=None,
    tokenizer=tokenizer
)
print(train_dataset.max_length)

val_dataset = SpamDataset(
    csv_file="validation.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)
test_dataset = SpamDataset(
    csv_file="test.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)

[50256]
120


In [34]:
from torch.utils.data import DataLoader
num_workers = 0
batch_size = 8
torch.manual_seed(123)
# This setting ensures compatibility
# with most computers.
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False
)

In [35]:
input_batch, target_batch  = next(iter(train_loader))

print("Input batch dimensions:", input_batch.shape)
print("Label batch dimensions", target_batch.shape)

Input batch dimensions: torch.Size([8, 120])
Label batch dimensions torch.Size([8])


In [42]:
from pathlib import Path

import tiktoken
import torch

from config import HF_MODELS, INSTRUCTION_DATA_DIR, MODEL_CONFIG, VARIANT
from data.dataset import data_split, get_instruction_loaders
from inference.load_weights import load_from_hf
from model.gpt import GPTModel
from finetune.instructure_follower_finetuning import loading_model

ImportError: cannot import name 'BASE_CONFIG' from 'config' (/content/miniGPT/config.py)

In [37]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Loading the model

In [ ]:
def loading_model(variant: str) -> tuple[torch.nn.Module, dict]:
    """Load a model from HuggingFace or a local checkpoint."""
    if variant in HF_MODELS:
        print(f"Loading pretrained model from HuggingFace: {variant}")
        model, config = load_from_hf(variant)
    else:
        print(f"Loading model from checkpoint: {variant}")
        model = GPTModel(MODEL_CONFIG).to(device)
        model.load_state_dict(torch.load(variant, map_location=device))
        config = MODEL_CONFIG
    return model.to(device), config

Prepare the model for classification

In [ ]:
def setup_classification_model(variant: str, num_classes: int = 2) -> tuple[torch.nn.Module, dict]:
    """Load a pretrained GPT and adapt it for sequence classification."""
    model, config = loading_model(variant)

    param_count = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {param_count:,}")

    # Freeze all pretrained weights
    for param in model.parameters():
        param.requires_grad = False

    # Replace LM head with a classification head
    torch.manual_seed(123)
    model.out_head = torch.nn.Linear(
        in_features=model.out_head.in_features,
        out_features=num_classes,
    )

    # Unfreeze last transformer block and final norm for fine-tuning
    for param in model.trf_blocks[-1].parameters():
        param.requires_grad = True
    for param in model.final_norm.parameters():
        param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {trainable:,}")

    return model, config

#for modal use the following lines to call it
#model, config = setup_classification_model(VARIANT)
#model.to(device)

In [51]:
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
   model.eval()
   correct_predictions, num_examples = 0, 0
   if num_batches is None:
       num_batches = len(data_loader)
   else:
       num_batches = min(num_batches, len(data_loader))
   for i, (input_batch, target_batch) in enumerate(data_loader):
       if i < num_batches:
           input_batch, target_batch = input_batch.to(device), target_batch.to(device)
           with torch.no_grad():
               logits = model(input_batch)[:, -1, :]             #A
           predicted_labels = torch.argmax(logits, dim=-1)
           num_examples += predicted_labels.shape[0]
           correct_predictions += (predicted_labels == target_batch).sum().item()
       else:
           break
   return correct_predictions / num_examples